* [#29](https://github.com/salgo60/ifkdb/issues/29)
* denna notebook [29_linkroot_ifkdb.ipynb](29_linkroot_ifkdb.ipynb)


In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-03-03 09:38:58.802053


In [2]:
import requests

def check_url(url):
    r = requests.get(url, timeout=10, allow_redirects=True)
    
    final_url = r.url
    status = r.status_code
    
    # Hard 404
    if status == 404:
        return "hard 404", final_url
    
    # Redirect to homepage = broken routing
    if final_url.rstrip("/") == "https://ifkdb.se":
        return "redirect to homepage (broken)", final_url
    
    # Soft 404 check (if site uses text message)
    if status == 200 and "hittas inte" in r.text.lower():
        return "soft 404", final_url
    
    # Looks OK only if still on /spelare/
    if "/spelare/" in final_url:
        return "ok", final_url
    
    return f"other({status})", final_url


base = "https://ifkdb.se/spelare/"
id = "SamuelWowoah_840"

url1 = base + id
print(check_url(url1))

url2 = url1.replace("SamuelWowoah","Samuel-Wowoah")
print(check_url(url2))

('redirect to homepage (broken)', 'https://ifkdb.se/')
('ok', 'https://ifkdb.se/spelare/Samuel-Wowoah_840')


In [ ]:
from tqdm import tqdm
import requests
import pandas as pd
import time
from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# -------------------------
# 1. SPARQL QUERY
# -------------------------

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

query = """
SELECT ?item ?itemLabel ?ifkdbid ?ifkdb WHERE {
  ?item wdt:P11905 ?ifkdbid.
  BIND(URI(CONCAT("https://ifkdb.se/spelare/", ?ifkdbid)) AS ?ifkdb)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "sv,en,mul". }
}
"""

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "IFKDB-Link-Checker/1.0 (research QA)"
}

r = requests.get(SPARQL_ENDPOINT, params={"query": query}, headers=headers)
data = r.json()

rows = []
for item in data["results"]["bindings"]:
    rows.append({
        "label": item.get("itemLabel", {}).get("value", ""),
        "ifkdb": item["ifkdb"]["value"]
    })

df = pd.DataFrame(rows)

# -------------------------
# 2. Friendly session setup
# -------------------------

session = requests.Session()

retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("http://", adapter)
session.mount("https://", adapter)

session.headers.update({
    "User-Agent": "IFKDB-Link-Checker/1.0 (contact: research QA bot)"
})

# -------------------------
# 3. URL CHECK FUNCTION
# -------------------------

def check_url(url):
    try:
        r = session.get(url, timeout=8, allow_redirects=True)
        final_url = r.url
        status = r.status_code

        if status == 404:
            return "hard 404", final_url
        
        if final_url.rstrip("/") == "https://ifkdb.se":
            return "redirect to homepage", final_url

        if status == 200 and "/spelare/" in final_url:
            return "ok", final_url

        return f"other({status})", final_url

    except Exception as e:
        return f"error", ""

# -------------------------
# 4. LOOP (with delay)
# -------------------------

results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking IFKDB URLs"):
    status, final_url = check_url(row["ifkdb"])
    
    if status != "ok":
        results.append({
            "Player": row["label"],
            "Original URL": row["ifkdb"],
            "Final URL": final_url,
            "Status": status
        })

    time.sleep(0.5) 
error_df = pd.DataFrame(results)

# -------------------------
# 5. Make links clickable
# -------------------------

def make_clickable(url):
    if url:
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""

if not error_df.empty:
    error_df["Original URL"] = error_df["Original URL"].apply(make_clickable)
    error_df["Final URL"] = error_df["Final URL"].apply(make_clickable)

# -------------------------
# 6. HTML REPORT
# -------------------------

today = datetime.now().strftime("%Y%m%d")
filename = f"ifkdb_linkroot_report_{today}.html"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

html = f"""
<html>
<head>
<meta charset="utf-8">
<title>IFKDB Länkrapport</title>
<style>
body {{ font-family: Arial; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #ccc; padding: 6px; }}
th {{ background-color: #f2f2f2; }}
tr:nth-child(even) {{ background-color: #fafafa; }}
a {{ text-decoration: none; color: blue; }}
</style>
</head>
<body>
<h1>IFKDB Länkrapport</h1>
<p>Genererad: {timestamp}</p>
<p>Antal fel: {len(error_df)}</p>
{error_df.to_html(index=False, escape=False)}
</body>
</html>
"""

with open(filename, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Rapport skapad: {filename}")

Checking IFKDB URLs:  93%|██████████████████▌ | 539/582 [17:56<01:26,  2.01s/it]

In [ ]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))
